In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.transforms as mtransforms
import matplotlib.colors as mcolors
from openpmd_viewer import OpenPMDTimeSeries
from scipy.ndimage import gaussian_filter
import scipy.constants as sc

In [ ]:
def add_external_parameter_box(fig=None, params=None, theme='neutral', ax=None, dx=0, dy=0):
    """
    Places a text box outside the plot and dynamically forces the 
    graph to shrink just enough to prevent clipping or overlap.
    """
    if fig is None:
        fig = plt.gcf()
    if params is None:
        params = {"Status": "Default"}
        
    # 1. Themes definition
    themes = {
        'neutral':  {'bg': '#f8f9fa', 'border': '#cccccc', 'text': '#333333'},
        'success':  {'bg': '#e6f4ea', 'border': '#137333', 'text': '#137333'},
        'warning':  {'bg': '#fef7e0', 'border': '#b06000', 'text': '#b06000'},
        'critical': {'bg': '#fce8e6', 'border': '#c5221f', 'text': '#c5221f'}
    }
    style = themes.get(theme, themes['neutral'])
    text_str = []
    for key, val in params.items():
        if val > 1000:
            text_str.append(f"{key}: {val:.2e}")
        else:
            text_str.append(f"{key}: {val}")
    text_str = "\n".join(text_str)
    
    # Place text anchored to the right side of the plot box
    if fig:
        t = fig.text(
            0.1 + dx, 0.5 + dy, text_str, fontsize=10, color=style['text'], weight='medium',
            verticalalignment='center', horizontalalignment='left', transform=fig.axes[1].transAxes,
            bbox=dict(boxstyle="round,pad=0.6", facecolor=style['bg'], edgecolor=style['border'], linewidth=1.5)
        )
    elif ax:
        t = ax.text(
                    0.1 + dx, 0.5 + dy, text_str, fontsize=10, color=style['text'], weight='medium',
                    verticalalignment='center', horizontalalignment='left', transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.6", facecolor=style['bg'], edgecolor=style['border'], linewidth=1.5)
                )

In [ ]:
from coil_3d import SingleCoil3DConfig
cfg = SingleCoil3DConfig()
static_params = {
    "$n_{stream}$": cfg.n_stream,
    "$v_{drift} (m/s)$": cfg.v_drift,
    "$T_{i}(eV)$": cfg.T_i_eV,
    "$T_{e}(eV)$": cfg.T_e_eV,
    "I(A)": cfg.I,
    "R(m)": cfg.R_coil,
}

In [ ]:
MU0  = sc.mu_0
gamma = 5/3
Ti_eV = cfg.T_i_eV
Te_eV = cfg.T_e_eV
Ti_J = Ti_eV * sc.eV
Te_J = Te_eV * sc.eV
n_stream = cfg.n_stream
v_drift = cfg.v_drift
rho = n_stream * sc.m_p
P_ram  = rho * v_drift**2
P_th = n_stream * (Ti_J + Te_J)
P_tot = P_ram + P_th
I = cfg.I
dia = cfg.dia
R_coil = dia / 2
r_CF = np.sqrt((MU0 * I * R_coil**2 / (2 * np.sqrt(2 * MU0 * P_tot)))**(2/3) - R_coil**2)
print(r_CF)
save_path = './diags'
series_f   = OpenPMDTimeSeries(save_path + '/field_diag')
series_p   = OpenPMDTimeSeries(save_path + '/part_diag')
iterations = series_f.iterations[1:]



# Velocity moments

In [ ]:
def bin_moments(x, z, vx, vy, vz, w, xs, zs):
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    x_edges = np.r_[xs - 0.5*dx, xs[-1] + 0.5*dx]
    z_edges = np.r_[zs - 0.5*dz, zs[-1] + 0.5*dz]

    counts, _, _ = np.histogram2d(x, z, bins=[x_edges, z_edges], weights=w)

    safe = np.where(counts > 0, counts, 1.0)

    def wmean(v):
        h, _, _ = np.histogram2d(x, z, bins=[x_edges, z_edges], weights=w*v)
        return h / safe

    vx_m, vy_m, vz_m = wmean(vx), wmean(vy), wmean(vz)

    # per-cell variance requires per-particle subtraction of the cell mean
    # need to map each particle to its cell mean first
    ix = np.clip(np.searchsorted(x_edges, x) - 1, 0, len(xs)-1)
    iz = np.clip(np.searchsorted(z_edges, z) - 1, 0, len(zs)-1)

    dvx = vx - vx_m[ix, iz]
    dvy = vy - vy_m[ix, iz]
    dvz = vz - vz_m[ix, iz]

    sigma2_raw, _, _ = np.histogram2d(x, z, bins=[x_edges, z_edges],
                                    weights=w*(dvx**2 + dvy**2 + dvz**2))
    sigma2 = np.where(counts > 0, sigma2_raw / safe, 0.0)

    return sigma2, [vx_m, vy_m, vz_m]

# optimized velocity moments

In [ ]:
def bin_moments(x, z, vx, vy, vz, w, xs, zs):
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    x_edges = np.r_[xs - 0.5*dx, xs[-1] + 0.5*dx]
    z_edges = np.r_[zs - 0.5*dz, zs[-1] + 0.5*dz]
    # one-time cell index per particle, flattened for bincount
    ix = np.clip(np.searchsorted(x_edges, x) - 1, 0, len(xs)-1)
    iz = np.clip(np.searchsorted(z_edges, z) - 1, 0, len(zs)-1)
    flat = ix * len(zs) + iz
    nbins = len(xs) * len(zs)

    def cell_sum(vals):
        return np.bincount(flat, weights=w*vals, minlength=nbins).reshape(len(xs), len(zs))

    counts = cell_sum(np.ones_like(w))
    safe = np.where(counts > 0, counts, 1.0)

    # first and second moments in the same pass, no per-particle re-subtraction
    sum1 = {c: cell_sum(v) for c, v in zip('xyz', (vx, vy, vz))}
    sum2 = {c: cell_sum(v**2) for c, v in zip('xyz', (vx, vy, vz))}

    means = {c: sum1[c]/safe for c in 'xyz'}
    sigma2 = sum(sum2[c]/safe - means[c]**2 for c in 'xyz')
    sigma2 = np.where(counts > 0, sigma2, 0.0)

    vm = [means[c] for c in 'xyz']

    return sigma2, vm

# Beta map over time

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
beta_frames = []
xs = zs = None

sigma2s, vms = [], []

iterations = series_f.iterations

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho, _   = series_f.get_field('rho_stream_i', iteration=it, slice_across='y')
    #rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

    x, z, ux, uy, uz, w = series_p.get_particle(var_list=['x', 'z', 'ux', 'uy', 'uz', 'w'], iteration=it)

    xs, zs = info.x, info.z

    B2      = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
    n_stream_t       = np.abs(rho + 1e-12) / sc.e
    #n_bg_t = np.abs(rho_bg + 1e-12) / sc.e
    # ux = gamma * v / c
    # https://warpx.readthedocs.io/en/26.01/usage/parameters.html#particle-push-charge-and-current-deposition-field-gathering
    # under: <diag_name>.particle_fields.<field_name>(x,y,z,ux,uy,uz)
    vx, vy, vz = ux * sc.c, uy * sc.c, uz * sc.c
    sigma2, vm = bin_moments(x, z, vx, vy, vz, w, xs, zs)
    sigma2s.append(sigma2)
    vms.append(vm)
    p_i = (1/3) * sc.m_p * (n_stream_t) *  sigma2
    p_e = n_stream * Te_J * (n_stream_t / n_stream)**(gamma)
    #p_bg = n_bg_t * (Ti_J) + n_bg_t * sc.m_p * cfg.v_drift**2
    v_bulk2 = vm[0]**2 + vm[1]**2 + vm[2]**2
    p_ram = n_stream_t * sc.m_p * v_bulk2
    p_tot = (p_i + p_e + p_ram)
    # P_tot = P_ram + P_th = n_stream(v_drift**2 + Ti_J + Te_J)
    beta_th = (p_tot) / (B2 / (2 * MU0))

    beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))
    #beta_frames.append(np.log10(np.maximum(beta_th, 1e-12)))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=(2, 1))
ax_text.axis('off')
im = ax.imshow(beta_frames[0], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal')
contour_handle = [ax.contour(xs, zs, beta_frames[0], levels=[0.0],
                             colors='k', linewidths=1.0)]
plt.colorbar(im, ax=ax, label='log₁₀ β_th')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('HELLO IM" TITLE')

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    it = iterations[k]
    im.set_data(beta_frames[k])
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(xs, zs, beta_frames[k], levels=[0.0],
                                   colors='k', linewidths=1.0)
    t_us = series_f.t[k] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/beta_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_timelapse.mp4")

# B Lineout over time

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
B_lineouts = []
xs_line    = None

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'])

    if xs_line is None:
        xs_line = info.x

    B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    B_lineouts.append(gaussian_filter(B_mag, sigma=1))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
B_max = max(b.max() for b in B_lineouts)
print(B_max)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1), layout='constrained')
ax_text.axis('off')
line,  = ax.plot(xs_line, B_lineouts[0], color='steelblue', lw=2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylim(0, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('|B| (T)')
title = ax.set_title('')

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    line.set_ydata(B_lineouts[k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'|B| lineout  step {iterations[k]}  t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/B_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/B_lineout_timelapse.mp4")



# Streamplots

In [ ]:
# Precompute Streamplots
B_streamplots = []
xs_line = None
xz_line = None
iterations = series_f.iterations[1:]
for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')
    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y'])

    xs_line = info.x
    xz_line = info.z 

    B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    B_streamplots.append([Bx, Bz, B_mag])

print("\nDone")

# Animate
B_max = max(b[2].max() for b in B_streamplots)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=[2, 1])
ax_text.axis('off')
ax.set_aspect('equal')
vmax = np.max([b[2] for b in B_streamplots])
vmin = np.min([b[2] for b in B_streamplots])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)  # Explicitly enforces 0 as white across all frames
im = ax.streamplot(
    x=info.x,
    y=info.z,
    u=B_streamplots[0][0],
    v=B_streamplots[0][1],
    color=B_streamplots[0][2], norm=norm,
    cmap="viridis", density=2.0, linewidth=1,
    broken_streamlines=True,
)

cbar = fig.colorbar(im.lines, ax=ax, label=r'$|B|$')

ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('Iteration 0')

add_external_parameter_box(fig, static_params)

plt.savefig('test.png')
plt.show()

def update(k):
    it = iterations[k]
    ax.cla()
    im = ax.streamplot(
        x=info.x,
        y=info.z,
        u=B_streamplots[k][0],
        v=B_streamplots[k][1],
        color=B_streamplots[k][2],
        cmap="viridis", density=2.0, linewidth=1
    )
    t_us = series_f.t[k] * 1e6
    ax.set_title(f'|B|  step {it}  t = {t_us:.2f} µs')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('z (m)')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/B_stream_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/B_stream_timelapse.mp4")

# Face Cusp Losses over time (make a copy if running mid run)

In [ ]:
data   = np.load(f'diags/cusp_flux.npz', allow_pickle=True)
times  = data['times'] * 1e6          # convert to µs
flux_minus = data['flux_minus']               # shape (n_steps, 6)
flux_plus = data['flux_plus']
#labels = data['face_labels']

print(flux_plus.shape)
print(flux_minus.shape)

flux_minus = flux_minus.T

# Total flux across various radii
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1))
ax_text.axis('off')

rs = np.linspace(cfg.R_coil / flux_minus.shape[0], cfg.R_coil, flux_plus.shape[1])

print(rs.shape)

lines = []
for k, r in enumerate(rs):
    if k > 0:
        line, = ax.plot(times, flux_minus[k] - flux_minus[k-1], lw = 1.5, label=f'leaving, {0.1*(k):.2f} <= r < {r:.2f}')
    else:
        line, = ax.plot(times, flux_minus[k], lw = 1.5, label=f'leaving, r < {r:.2f}')
    lines.append(line)

line, = ax.plot(times, flux_minus[-1], lw=1.5, label=f'total')
lines.append(line)

ax.set_xlabel('time (µs)')
ax.set_ylabel('$log_{10}$(particles/s)')
ax.set_title('Cusp loss rate vs time')

labels = [h.get_label() for h in lines]
ax_text.legend(lines, labels)

add_external_parameter_box(fig, static_params, dx=-0.4, dy=-0.2)
plt.tight_layout()
plt.savefig(f'diags/cusp_loss_total.png', dpi=150)
plt.show()

# ax.plot(times, np.log10(flux_minus), color='steelblue', lw=1.5, label='leaving')
# # ax.plot(times, np.log10(flux_plus), color='red', lw=1.5, label='returning')
# ax.set_xlabel('time (µs)')
# ax.set_ylabel('$log_{10}$(particles/s)')
# ax.set_title('Cusp loss rate vs time')

# total_loss = flux_minus

# # ── Total loss rate vs time ───────────────────────────────────────
# fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1))
# ax_text.axis('off')
# ax.plot(times, np.log10(total_loss), color='steelblue', lw=1.5, label='leaving')
# ax.plot(times, np.log10(flux_plus), color='red', lw=1.5, label='returning')
# ax.set_xlabel('time (µs)')
# ax.set_ylabel('$log_{10}$(particles/s)')
# ax.set_title('Cusp loss rate vs time')
# ax.legend()
# add_external_parameter_box(fig, static_params)
# plt.tight_layout()
# plt.savefig(f'diags/cusp_loss_total.png', dpi=150)
# plt.show()

# Streamplot on top of beta

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import scipy.constants as sc
from scipy.ndimage import gaussian_filter
from openpmd_viewer import OpenPMDTimeSeries

MU0 = sc.mu_0

# ── Configuration ────────────────────────────────────────────────
save_path  = "diags"

# ── Merged Precompute Loop ───────────────────────────────────────
#beta_frames  = []
# B_streamplots = []
# xs = zs = None

# for k, it in enumerate(iterations[1:]):
#     print(f"  Loading {k+1}/{len(iterations)-1}", end='\r')

#     Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
#     By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
#     Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
#     rho,    _ = series_f.get_field('rho_stream_i',     iteration=it, slice_across='y')
#     rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

#     if xs is None:
#         xs, zs = info.x, info.z

#     # Beta frame
#     B2     = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
#     n      = np.abs(rho + rho_bg) / sc.e
#     Ti_J   = Ti_eV * sc.eV
#     beta_th = n * Ti_J / (B2 / (2 * MU0))
#     #beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))

#     # Streamplot frame (Bx, Bz in the xz plane)
#     B_mag = np.sqrt(Bx**2 + Bz**2)
#     B_streamplots.append((Bx, Bz, B_mag))

# print("\nDone.")

# ── Helper: remove streamplot artists ───────────────────────────
def _clear_streamplot(sp):
    sp.lines.remove()


# ── Figure Setup ─────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
ax_text.axis('off')
ax2 = ax.inset_axes([0, 0, 1, 1])
ax2.set_axis_off()
ax2.patch.set_alpha(0)

# Layer 0: beta heatmap
im = ax.imshow(
    beta_frames[0], origin='lower',
    extent=[xs[0], xs[-1], zs[0], zs[-1]],
    vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal', zorder=0
)

# Layer 1: beta=1 contour
contour_handle = [ax.contour(
    xs, zs, beta_frames[0], levels=[0.0],
    colors='k', linewidths=1.0, zorder=1
)]

vmin_stream = np.min([s[2] for s in B_streamplots])
vmax_stream = np.max([s[2] for s in B_streamplots])
norm = mcolors.Normalize(vmin=vmin_stream, vmax=vmax_stream)  # Explicitly enforces 0 as white across all frames

# Layer 2: B-field streamplot on top
stream_handle = [ax2.streamplot(
    xs, zs,
    B_streamplots[0][0], B_streamplots[0][1], norm=norm,
    color=B_streamplots[0][2], cmap='hot', density=1.2, linewidth=0.8, zorder=2,
)]

plt.colorbar(im, ax=ax, label=r'$log_{10}$ $\beta$', pad=0.1)
plt.colorbar(stream_handle[0].lines, ax=ax, label='|B|', pad=0.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.legend(loc='upper right', fontsize=8)
title = ax.set_title('')

add_external_parameter_box(fig, static_params)

plt.savefig('test.png')
plt.show()

# ── Animation Update ─────────────────────────────────────────────
def update(k):
    it = iterations[k]
    ax2.cla()
    ax2.set_axis_off()
    ax2.patch.set_alpha(0)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_ylim(ax.get_ylim())

    # Update beta heatmap
    im.set_data(beta_frames[k])

    # Update beta=1 contour
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(
        xs, zs, beta_frames[k], levels=[0.0],
        colors='k', linewidths=1.0, zorder=1
    )

    stream_handle[0] = ax2.streamplot(
        xs, zs,
        B_streamplots[k][0], B_streamplots[k][1],
        color=B_streamplots[k][2], cmap='hot',  density=1.2, linewidth=0.8, zorder=2,
    )

    t_us = series_f.t[k] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

# ── Save ─────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=len(iterations), interval=100
)
ani.save(f'{save_path}/beta_stream_overlay.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_stream_overlay.mp4")

# Bx lineout 

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
B_lineouts = []
xs_line    = None

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'])

    if xs_line is None:
        xs_line = info.x

    # B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    # B_lineouts.append(gaussian_filter(B_mag, sigma=1))
    B_lineouts.append(Bx)

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
B_max = max(b.max() for b in B_lineouts)
B_min = min(b.min() for b in B_lineouts)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1), layout='constrained')
ax_text.axis('off')
line,  = ax.plot(xs_line, B_lineouts[0], color='steelblue', lw=2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylim(B_min * 1.1, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('Bx (T)')
title = ax.set_title('')

add_external_parameter_box(fig, static_params)

def update(k):
    line.set_ydata(B_lineouts[k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'Bx lineout  step {iterations[k]}  t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Bx_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/Bx_lineout_timelapse.mp4")

# Cusp losses time-synced

In [ ]:
data   = np.load(f'{save_path}/cusp_flux.npz', allow_pickle=True)
times  = data['times'] * 1e6          # convert to µs
flux_minus = data['flux_minus']               # shape (n_steps, 6)
flux_plus = data['flux_plus']
#flux_minus_upstream = data['flux_minus_upstream']
#labels = data['face_labels']

total_loss = flux_minus.T[0]
t_k = [None]
text = [None]
# ── Total loss rate vs time ───────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1))
ax_text.axis('off')
ax.plot(times, total_loss, color='steelblue', lw=1.5, label='Flux through coil @ x = 0.0')
#ax.plot(times, flux_minus_upstream, color='c', lw=1.5, label='Upstream Flux @ x = 2.0')
#ax.axhline(np.mean(flux_minus_upstream[20:]), label="Upstream flow average", color='g', linestyle='--')
#ax.text(0, np.mean(flux_minus_upstream[20:]), f'{np.mean(flux_minus_upstream[20:]):.2e}')
diag_times = series_f.t * 1e6
print(len(diag_times), len(total_loss))
t_k[0] = ax.axvline(diag_times[0], color='red')
ax.set_xlabel('time (µs)')
ax.set_ylabel('particles lost per step')
ax.set_title('Cusp loss rate vs time')
text[0] = ax.text(diag_times[0], total_loss[0], f"{total_loss[0]:.2e}")
# ax.legend(loc='upper right')
add_external_parameter_box(fig, static_params)
plt.tight_layout()

def update(k):
    it = iterations[k]
    t = series_f.t[k] * 1e6
    title = f"Cusp loss rate it: {it} t: {t:.2f} us"
    ax.set_title(title)
    t_k[0].remove()
    t_k[0] = ax.axvline(diag_times[k], color='red')
    text[0].remove()
    xlim = ax.get_xlim()
    frac = (diag_times[k] - xlim[0]) / (xlim[1] - xlim[0])
    ha = 'right' if frac > 0.7 else 'left'
    offset = -0.02 * (xlim[1] - xlim[0]) if ha == 'right' else 0.02 * (xlim[1] - xlim[0])
    try:
        idx = np.searchsorted(times, diag_times[k])
        text[0] = ax.text(diag_times[k] + offset, total_loss[idx], f"{total_loss[idx]:.2e}", ha=ha)
    except:
        text[0] = ax.text(diag_times[-1] + offset, total_loss[-1], f"{total_loss[-1]:.2e}", ha=ha)

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/cusp_loss_time_synced.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/cusp_loss_time_synced.mp4")

# Density over B-lines

In [ ]:
MU0 = sc.mu_0

# ── Merged Precompute Loop ───────────────────────────────────────
density_frames  = []
# B_streamplots = []
# xs = zs = None
w = series_p.get_particle(var_list=['w'], iteration=iterations[5])
for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho,    _ = series_f.get_field('rho_stream_i',     iteration=it, slice_across='y')
    
    xs, zs = info.x, info.z

    # Density
    density_frames.append(np.log10((rho + 1e-12) / sc.elementary_charge))

print("\nDone.")

# ── Helper: remove streamplot artists ───────────────────────────
def _clear_streamplot(sp):
    sp.lines.remove()


# ── Figure Setup ─────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
ax_text.axis('off')
ax2 = ax.inset_axes([0, 0, 1, 1])
ax2.set_axis_off()
ax2.patch.set_alpha(0)

vmin_density = np.min([d for d in density_frames])
vmax_density = np.max([d for d in density_frames])

vmin_stream = np.min([s[2] for s in B_streamplots])
vmax_stream = np.max([s[2] for s in B_streamplots])
norm = mcolors.Normalize(vmin=vmin_stream, vmax=vmax_stream)

# Layer 0: beta heatmap
im = ax.imshow(
    density_frames[0], origin='lower',
    extent=[xs[0], xs[-1], zs[0], zs[-1]],
    vmin=vmin_density, vmax=vmax_density, cmap='plasma', aspect='equal', zorder=0
)

# Layer 2: B-field streamplot on top
stream_handle = [ax2.streamplot(
    xs, zs,
    B_streamplots[0][0], B_streamplots[0][1], norm = norm,
    color=B_streamplots[0][2], cmap='hot', density=1.2, linewidth=0.8, zorder=2,
)]

cb_density = plt.colorbar(im, ax=ax, label=r'$log_{10}$ $\rho$', pad=0.15)
cb_stream = plt.colorbar(stream_handle[0].lines, ax=ax, label='|B|', pad=0.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.legend(loc='upper right', fontsize=8)
title = ax.set_title('')

add_external_parameter_box(fig, static_params)
plt.savefig('test.png')
plt.show()

# ── Animation Update ─────────────────────────────────────────────
def update(k):
    it = iterations[k]
    ax2.cla()
    ax2.set_axis_off()
    ax2.patch.set_alpha(0)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_ylim(ax.get_ylim())

    # Update density map
    im.set_data(density_frames[k])
    im.set_clim(np.min(density_frames[k]), np.max(density_frames[k]))

    stream_handle[0] = ax2.streamplot(
        xs, zs,
        B_streamplots[k][0], B_streamplots[k][1],
        color=B_streamplots[k][2], cmap='hot',  density=1.2, linewidth=0.8, zorder=2,
    )

    t_us = series_f.t[k] * 1e6
    title.set_text(r'$\rho' + f' step {it}  t = {t_us:.2f} µs')
    return [im]

# ── Save ─────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=len(iterations), interval=100
)
ani.save(f'{save_path}/density_stream_overlay.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/density_stream_overlay.mp4")

# Current Jy on xz-plane @ y = 0

Note that $x \times z = -y$ and hence $sign(J_y) < 0 \rightarrow \odot$ direction, while $sign(J_y) > 0 \rightarrow \oplus$ direction

In [ ]:
J_s = []
xs, zs = None, None
for it in iterations:
    print(f"{it} / {iterations[-1]}", end="\r")
    Jx_ion, info = series_f.get_field(field='j', coord='x', slice_across='y', iteration=it)
    Jy_ion, info = series_f.get_field(field='j', coord='y', slice_across='y', iteration=it)
    Jz_ion, info = series_f.get_field(field='j', coord='z', slice_across='y', iteration=it)
    Jx_ele, _ = series_f.get_field(field='j_displacement', coord='x', slice_across='y', iteration=it)
    Jy_ele, _ = series_f.get_field(field='j_displacement', coord='y', slice_across='y', iteration=it)
    Jz_ele, _ = series_f.get_field(field='j_displacement', coord='z', slice_across='y', iteration=it)

    Jx = Jx_ion + Jx_ele
    Jy = Jy_ion + Jy_ele 
    Jz = Jz_ion + Jz_ele

    J_mag = np.sqrt(Jx**2 + Jy**2 + Jz**2)

    if xs is None:
        xs = info.x
        zs = info.z

    J_s.append([Jx, Jy, Jz, J_mag])

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
ax_text.axis('off')
vmin = np.min([s[1] for s in J_s])
vmax = np.max([s[1] for s in J_s])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
im = ax.imshow(J_s[0][1], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               cmap='RdBu_r', aspect='equal', vmin=vmin, vmax=vmax)


plt.colorbar(im, ax=ax, label='$J_{y}$')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('J_y on xz-plane')

add_external_parameter_box(fig, static_params)

ax_text.text(
    0.1, 0.9, r'$-y: \odot$' + '\n' + r'$+y: \oplus$', fontsize=15, weight='medium',
    verticalalignment='center', horizontalalignment='left', transform=fig.axes[1].transAxes,
    bbox=dict(boxstyle="round,pad=0.6", facecolor='#f8f9fa', linewidth=1.5)
)

plt.savefig('test.png')
plt.show()

def update(k):
    it = iterations[k]
    im.set_data(J_s[k][1])

    t_us = series_f.t[k] * 1e6
    title.set_text(f'Jy on xz-plane  step {it}  t = {t_us:.2f} µs')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Jy_xz_plane_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/Jy_xz_plane_timelapse.mp4")

# $J_\theta(y, z)$

## 2D map of azimuthal (ring-current) component of plasma currents

### What it represents
- Each pixel is local current vector, tangential direction around x-axis
    - Circulating like a ring around a coil

### Azimuthal instead of cartesian
- Isolation of circular motion about the x-axis (to identify diamagnetic effects of gyromotion)

### +/- signs
- Positive: same rotation as coil (paramagnetic)
- Negative: opposing (diamagnetic)
- Where are the coil/plasma current standoffs occurring 

### Separating protons, electrons
- Influenced differently, with differing diamagnetic effects
- Understand fluid vs kinetic contributions

### Limitations
- Single x-value (@ x = 0)

In [ ]:
J_thetas = []

J_thetas = {
    'e': [],
    'p': [],
    't': []
}


for it in iterations:
    print(f"Processed: {it} / {iterations[-1]}", end='\r')
    Jx_ion, info = series_f.get_field(field='j', coord='x', slice_across='x', iteration=it)
    Jy_ion, info = series_f.get_field(field='j', coord='y', slice_across='x', iteration=it)
    Jz_ion, info = series_f.get_field(field='j', coord='z', slice_across='x', iteration=it)
    Jx_ele, _ = series_f.get_field(field='j_displacement', coord='x', slice_across='x', iteration=it)
    Jy_ele, _ = series_f.get_field(field='j_displacement', coord='y', slice_across='x', iteration=it)
    Jz_ele, _ = series_f.get_field(field='j_displacement', coord='z', slice_across='x', iteration=it)

    Jx = Jx_ion + Jx_ele
    Jy = Jy_ion + Jy_ele 
    Jz = Jz_ion + Jz_ele

    ys, zs = info.y, info.z

    # meshgrid ordering matches existing field array shape: rows=z, cols=y
    Z, Y = np.meshgrid(zs, ys, indexing='ij')
    r = np.sqrt(Y**2 + Z**2)

    # mask the on-axis singularity (r->0) rather than letting it blow up
    #r_safe = np.where(r < 2 * (ys[1] - ys[0]), np.nan, r)  # ~2 cells around axis

    J_theta_tot = (Jz * Y - Jy * Z) / r
    J_theta_e = (Jz_ele * Y - Jy_ele * Z) / r
    J_theta_p = (Jz_ion * Y - Jy_ion * Z) / r

    J_thetas['t'].append(J_theta_tot)
    J_thetas['e'].append(J_theta_e)
    J_thetas['p'].append(J_theta_p)

species_to_title = {
    'e': r'$J_e$',
    'p': r'$J_p$',
    't': r'$J_e + J_p$'
}

for species in ['t', 'e', 'p']:

    fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
    ax_text.axis('off')
    # ax_text.text(
    #     0.1, 0.9, r'$+y: \odot$' + '\n' + r'$-y: \oplus$', fontsize=15, weight='medium',
    #     verticalalignment='center', horizontalalignment='left', transform=fig.axes[1].transAxes,
    #     bbox=dict(boxstyle="round,pad=0.6", linewidth=1.5)
    # )
    # fixed vmin and vmax
    vmax = np.nanmax([np.abs(i) for i in J_thetas[species]])
    vmin = -vmax
    im.set_clim(vmin=-vmax, vmax=vmax)

    im = ax.imshow(J_thetas[species][0], origin='lower',
                extent=[ys[0], ys[-1], zs[0], zs[-1]],
                cmap='RdBu_r', aspect='equal',
                vmin = vmin, vmax=vmax)

    plt.colorbar(im, ax=ax, label=r'$J_{\theta}$')
    ax.set_xlabel('y (m)')
    ax.set_ylabel('z (m)')
    title = ax.set_title(r'$J_{\theta}$ on yz-plane' + f', species: {species_to_title[species]}')

    add_external_parameter_box(fig, static_params)
    plt.savefig(f'test_{species}.png')
    plt.show()

    def update(k):
        it = iterations[k]
        im.set_data(J_thetas[species][k])
        vmin = np.min(J_thetas[species][k])
        vmax = np.max(J_thetas[species][k])

        t_us = series_f.t[k] * 1e6
        title.set_text(r'$J_{\theta}$ on yz-plane, ' + f'species: {species} it {it:.3f} t = {t_us:.2f} us')
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
    ani.save(f'{save_path}/Jtheta_{species}_yz_plane_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
    plt.close()
    print(f"Saved: {save_path}/Jtheta_{species}_yz_plane_timelapse.mp4")

# J streamplot on yz-plane

## Capturing:
- Closed loops about x -> azimuthal (ring currents) - diamagnetic/paramagnetic flow shapes
- Spirals in/out/radially -> compression/expansion

In [ ]:
ys, zs, r = None, None, None
Js = []
iterations = series_f.iterations
for it in iterations:
    print(f"{it}/{iterations[-1]}", end="\r")
    Jx_ion, info = series_f.get_field(field='j', coord='x', slice_across='x', iteration=it)
    Jy_ion, info = series_f.get_field(field='j', coord='y', slice_across='x', iteration=it)
    Jz_ion, info = series_f.get_field(field='j', coord='z', slice_across='x', iteration=it)
    Jx_ele, _ = series_f.get_field(field='j_displacement', coord='x', slice_across='x', iteration=it)
    Jy_ele, _ = series_f.get_field(field='j_displacement', coord='y', slice_across='x', iteration=it)
    Jz_ele, _ = series_f.get_field(field='j_displacement', coord='z', slice_across='x', iteration=it)

    Jx = Jx_ion + Jx_ele
    Jy = Jy_ion + Jy_ele 
    Jz = Jz_ion + Jz_ele

    J_mag = np.sqrt(Jy**2 + Jz**2)

    if ys is None:
        ys = info.y
        zs = info.z
        # meshgrid built once — reused every iteration since grid is static
        Z, Y = np.meshgrid(zs, ys, indexing='ij')
        r = np.sqrt(Y**2 + Z**2)

    # signed azimuthal current: + = same sense as coil (paramagnetic),
    # - = opposing sense (diamagnetic) — see established sign convention
    J_theta = (Jz * Y - Jy * Z) / r

    Js.append([Jy, Jz, J_theta])

# global symmetric color scale so 0 always maps to white and frames are
# comparable to each other (same reasoning as the J_theta imshow plots)

vmax = np.nanmax(np.abs([j[2] for j in Js]))
norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)  # Explicitly enforces 0 as white across all frames

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=[2, 1])
ax_text.axis('off')
im = ax.streamplot(
    x=ys,
    y=zs,
    u=Js[0][0],
    v=Js[0][1],
    color=Js[0][2],
    norm=norm,
    cmap="RdBu_r", density=2.0, linewidth=1,
    broken_streamlines=False,
)

cbar = fig.colorbar(im.lines, ax=ax, label=r'$J_\theta$')

ax.set_xlabel('y (m)')
ax.set_ylabel('z (m)')
# force fixed extent every frame — don't rely on streamplot's autoscale
ax.set_xlim(ys[0], ys[-1])
ax.set_ylim(zs[0], zs[-1])
ax.set_aspect('equal')  # streamplot resets this too on cla()
title = ax.set_title(f'J step 0 t = {series_f.t[0]*1e6} µs')

add_external_parameter_box(params=static_params, ax=ax_text)

plt.savefig('example_fig.png', )
plt.show()

def update(k):
    print(f"{k} / {len(iterations)}", end='\r')
    it = iterations[k]
    ax.cla()
    im = ax.streamplot(
        x=ys,
        y=zs,
        u=Js[k][0],
        v=Js[k][1],
        color=Js[k][2],
        norm=norm,
        cmap="RdBu_r", density=2.0, linewidth=1
    )
    # force fixed extent every frame — don't rely on streamplot's autoscale
    ax.set_xlim(ys[0], ys[-1])
    ax.set_ylim(zs[0], zs[-1])
    ax.set_aspect('equal')  # streamplot resets this too on cla()
    t_us = series_f.t[k] * 1e6
    ax.set_title(f'J  step {it}  t = {t_us:.2f} µs')
    ax.set_xlabel('y (m)')
    ax.set_ylabel('z (m)')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/J_stream_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/J_stream_timelapse.mp4")